# MCD per-scan per-class mean heights

Loads the JSON produced by `collect_per_scan_class_heights_mcd.py` and
plots how each class' mean height-above-per-scan-bottom varies across
scans. Heights use the first scan's lidar +z as the reference axis, with
each scan's bottom-most point as its local zero.

In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt

JSON_PATH = os.path.join(os.getcwd(), 'per_scan_class_heights_mcd.json')
with open(JSON_PATH) as f:
    data = json.load(f)

class_names = data['class_names']
scans = data['scans']
print(f"dataset={data['dataset']}  sequence={data.get('sequence_name')}  "
      f"scans={data['num_scans']}  up_ref={data['lidar_up_ref']}")

In [ ]:
# Reshape into per-class arrays aligned to scan order.
scan_idx = np.array([s['index'] for s in scans])
order = np.arange(len(scans))  # positional x-axis

per_class_mean = {name: np.full(len(scans), np.nan) for name in class_names}
per_class_count = {name: np.zeros(len(scans), dtype=np.int64) for name in class_names}
for i, s in enumerate(scans):
    for name, d in s['class_means'].items():
        per_class_mean[name][i] = d['mean']
        per_class_count[name][i] = d['count']

nonempty = [n for n in class_names if np.any(per_class_count[n] > 0)]
print('non-empty classes:', nonempty)

## Per-class mean height vs scan order
Each subplot is one class. The x-axis is scan position in the processed
sequence (after keyframe filtering). Gaps indicate scans that had no
points of that class.

In [ ]:
n = len(nonempty)
ncols = 2
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 2.6 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, name in zip(axes, nonempty):
    y = per_class_mean[name]
    ax.plot(order, y, lw=1.2, color='#3b82f6')
    finite = y[np.isfinite(y)]
    if finite.size:
        mu = float(finite.mean())
        ax.axhline(mu, color='#ef4444', ls='--', lw=1, label=f'overall mean = {mu:.2f} m')
        ax.legend(loc='upper right', fontsize=8)
    ax.set_title(f"{name}  (scans with class: {int(np.sum(per_class_count[name] > 0))})")
    ax.set_ylabel('mean h above bottom (m)')
    ax.grid(alpha=0.2)

for ax in axes[len(nonempty):]:
    ax.axis('off')

axes[-1].set_xlabel('scan order')
if len(nonempty) >= 2:
    axes[-2].set_xlabel('scan order')
plt.tight_layout()
plt.show()

## Overlay of all classes

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.get_cmap('tab10')
for i, name in enumerate(nonempty):
    ax.plot(order, per_class_mean[name], lw=1.3, color=cmap(i % 10), label=name)
ax.set_xlabel('scan order')
ax.set_ylabel('mean height above bottom (m)')
ax.set_title('Per-scan per-class mean height')
ax.grid(alpha=0.2)
ax.legend(loc='upper right', ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## Distribution of per-scan means
Each class gets a box plot over the set of per-scan mean heights (one
sample per scan that contains the class). This summarizes how stable the
per-scan mean is within a class.

In [ ]:
samples = [per_class_mean[name][np.isfinite(per_class_mean[name])] for name in nonempty]

fig, ax = plt.subplots(figsize=(9, 4.5))
bp = ax.boxplot(samples, labels=nonempty, showfliers=False, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#93c5fd')
    patch.set_edgecolor('#1e3a8a')
ax.set_ylabel('per-scan mean height above bottom (m)')
ax.set_title('Distribution of per-scan class-mean heights')
ax.grid(alpha=0.2, axis='y')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(f"{'class':<12} {'n_scans':>8} {'mean':>8} {'median':>8} {'std':>8} {'min':>8} {'max':>8}")
for name, arr in zip(nonempty, samples):
    if arr.size == 0:
        continue
    print(f"{name:<12} {arr.size:>8} {arr.mean():>8.2f} {np.median(arr):>8.2f} "
          f"{arr.std():>8.2f} {arr.min():>8.2f} {arr.max():>8.2f}")

## Per-class histogram of per-scan mean heights
One sample per scan that contains the class. Shared x-axis across all
classes so shapes are directly comparable.

In [ ]:
all_vals = np.concatenate([a for a in samples if a.size > 0]) if any(a.size for a in samples) else np.array([0.0])
x_lo = float(np.floor(all_vals.min()))
x_hi = float(np.ceil(all_vals.max()))
if x_hi <= x_lo:
    x_hi = x_lo + 1.0
bins = np.linspace(x_lo, x_hi, 41)

n = len(nonempty)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 2.6 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, name, arr in zip(axes, nonempty, samples):
    if arr.size == 0:
        ax.set_title(f"{name} (empty)")
        continue
    ax.hist(arr, bins=bins, color='#3b82f6', edgecolor='none', density=True)
    ax.axvline(arr.mean(), color='#ef4444', ls='--', lw=1.3, label=f'mean={arr.mean():.2f}')
    ax.axvline(np.median(arr), color='#10b981', ls=':', lw=1.3, label=f'median={np.median(arr):.2f}')
    ax.set_title(f"{name} (n={arr.size})")
    ax.set_xlabel('per-scan mean h (m)')
    ax.set_ylabel('density')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.2)

for ax in axes[len(nonempty):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## Overlay of per-class distributions
Normalized histograms on a single axis so per-class spreads can be
compared directly.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap('tab10')
for i, (name, arr) in enumerate(zip(nonempty, samples)):
    if arr.size == 0:
        continue
    counts, edges = np.histogram(arr, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(centers, counts, lw=1.6, color=cmap(i % 10), label=name)
    ax.fill_between(centers, counts, alpha=0.12, color=cmap(i % 10))
ax.set_xlabel('per-scan mean height above bottom (m)')
ax.set_ylabel('density')
ax.set_title('Per-scan class-mean height distributions (overlay)')
ax.grid(alpha=0.2)
ax.legend(loc='upper right', ncol=2, fontsize=9)
plt.tight_layout()
plt.show()